In [1]:
import pandas as pd

df = df = pd.read_csv("sample_10000.csv")
print(df.shape)
print(df.columns.tolist())

(10000, 56)
['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


In [2]:
df.head(10)

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1
5,23107.txt,https://www.globalreporting.org,30,www.globalreporting.org,23,0,org,100.0,1.000000,0.079963,...,0,0,1,35,1,11,86,0,14,1
6,23034.txt,https://www.saffronart.com,25,www.saffronart.com,18,0,com,100.0,1.000000,0.522907,...,0,0,1,32,4,14,44,2,17,1
7,696732.txt,https://www.nerdscandy.com,25,www.nerdscandy.com,18,0,com,100.0,1.000000,0.522907,...,0,0,1,24,2,22,36,0,15,1
8,739255.txt,https://www.hyderabadonline.in,29,www.hyderabadonline.in,22,0,in,100.0,1.000000,0.005084,...,0,0,1,71,4,9,40,1,317,1
9,14486.txt,https://www.aap.org,18,www.aap.org,11,0,org,100.0,1.000000,0.079963,...,0,0,1,10,1,12,173,6,65,1


In [3]:
print(df["label"].value_counts())

label
1    6149
0    3851
Name: count, dtype: int64


In [4]:
df["is_phishing"] = (df["label"] == 0).astype(int)

In [5]:
df[["URL", "Domain", "label", "is_phishing"]].head(10)

,URL,Domain,label,is_phishing
0,https://www.southbankmosaics.com,www.southbankmosaics.com,1,0
1,https://www.uni-mainz.de,www.uni-mainz.de,1,0
2,https://www.voicefmradio.co.uk,www.voicefmradio.co.uk,1,0
3,https://www.sfnmjournal.com,www.sfnmjournal.com,1,0
4,https://www.rewildingargentina.org,www.rewildingargentina.org,1,0
5,https://www.globalreporting.org,www.globalreporting.org,1,0
6,https://www.saffronart.com,www.saffronart.com,1,0
7,https://www.nerdscandy.com,www.nerdscandy.com,1,0
8,https://www.hyderabadonline.in,www.hyderabadonline.in,1,0
9,https://www.aap.org,www.aap.org,1,0


### Define function for retreve doman name

In [6]:
from urllib.parse import urlsplit

def extract_hostname(url):
    try:
        if not url.startswith(("http://", "https://")):
            url = "http://" + url

        return urlsplit(url).hostname
    except Exception:
        return None

In [7]:
import tldextract

def get_registered_domain(hostname):
    extracted = tldextract.extract(hostname)

    if extracted.domain and extracted.suffix:
        return f"{extracted.domain}.{extracted.suffix}"

    return hostname

In [8]:
test_url = "https://www.example.com/login/verify?id=123"

print(extract_hostname(test_url))

www.example.com


In [9]:
import dns.resolver

def get_ip_addresses(domain):
    try:
        answers = dns.resolver.resolve(domain, "A")
        return [answer.to_text() for answer in answers]
    except Exception:
        return []

In [10]:
domain = extract_hostname("https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset")

registered_domain = get_registered_domain(domain)

print(domain)
print(registered_domain)

print(get_ip_addresses(domain))

archive.ics.uci.edu
uci.edu
['128.195.10.252']


In [11]:
def get_name_servers(domain):
    try:
        answers = dns.resolver.resolve(domain, "NS")
        return [answer.to_text().rstrip(".") for answer in answers]
    except Exception:
        return []

In [12]:
print(get_name_servers(registered_domain))

['ns5.service.uci.edu', 'ns6.service.uci.edu']


### This function will add two new column for our dataset as ip_addresses and name_servers

In [13]:
def enrich_domain(domain, registered_domain):
    return {
        "ip_addresses": get_ip_addresses(domain),
        "name_servers": get_name_servers(registered_domain)
    }

In [14]:
domain = extract_hostname("https://www.example.com")
registered_domain = get_registered_domain(domain)

result = enrich_domain(domain, registered_domain)

print(result)

{'ip_addresses': ['172.66.147.243', '104.20.23.154'], 'name_servers': ['hera.ns.cloudflare.com', 'elliott.ns.cloudflare.com']}


### For testing we get only 100 records

In [15]:
test_df = df.head(100).copy()

test_df["domain"] = test_df["URL"].apply(extract_hostname)
test_df["registered_domain"] = test_df["domain"].apply(get_registered_domain)

test_df["ip_addresses"] = test_df["domain"].apply(get_ip_addresses)
test_df["name_servers"] = test_df["registered_domain"].apply(get_name_servers)

test_df[
    ["URL", "label", "domain", "ip_addresses", "name_servers"]
].head(20)

,URL,label,domain,ip_addresses,name_servers
0,https://www.southbankmosaics.com,1,www.southbankmosaics.com,[142.93.145.212],"[ns2.namebrightdns.com, ns1.namebrightdns.com]"
1,https://www.uni-mainz.de,1,www.uni-mainz.de,[134.93.178.47],"[b.ns14.net, c.ns14.net, ns-extern.zdv.Uni-Mai..."
2,https://www.voicefmradio.co.uk,1,www.voicefmradio.co.uk,"[216.137.52.12, 216.137.52.39, 216.137.52.128,...","[ns3.livedns.co.uk, ns2.livedns.co.uk, ns1.liv..."
3,https://www.sfnmjournal.com,1,www.sfnmjournal.com,"[162.159.140.114, 172.66.0.112]","[ns2.reedelsevier.com, ns1.reedelsevier.com, n..."
4,https://www.rewildingargentina.org,1,www.rewildingargentina.org,"[104.21.31.174, 172.67.178.236]","[hans.ns.cloudflare.com, lovisa.ns.cloudflare...."
5,https://www.globalreporting.org,1,www.globalreporting.org,"[172.66.159.113, 104.20.37.79]","[coco.ns.cloudflare.com, george.ns.cloudflare...."
6,https://www.saffronart.com,1,www.saffronart.com,"[104.26.8.7, 172.67.68.10, 104.26.9.7]","[romina.ns.cloudflare.com, damien.ns.cloudflar..."
7,https://www.nerdscandy.com,1,www.nerdscandy.com,"[104.18.12.41, 104.18.13.41]","[amit.ns.cloudflare.com, fish.ns.cloudflare.com]"
8,https://www.hyderabadonline.in,1,www.hyderabadonline.in,"[104.21.73.182, 172.67.165.62]","[yahir.ns.cloudflare.com, chan.ns.cloudflare.com]"
9,https://www.aap.org,1,www.aap.org,"[104.20.47.105, 172.66.148.195]","[ns4.p201.dns.oraclecloud.net, ns2.p201.dns.or..."


### Testing no of usable rows ( having both ip and name servers )

In [16]:
has_ip = test_df["ip_addresses"].apply(len) > 0

print("URLs with IP:", has_ip.sum())
print("Percentage:", has_ip.mean() * 100)

URLs with IP: 89
Percentage: 89.0


In [17]:
has_ns = test_df["name_servers"].apply(len) > 0

print("URLs with NS:", has_ns.sum())
print("Percentage:", has_ns.mean() * 100)

URLs with NS: 89
Percentage: 89.0


In [18]:
has_both = has_ip & has_ns

print("URLs with both:", has_both.sum())
print("Percentage:", has_both.mean() * 100)

URLs with both: 87
Percentage: 87.0


### Testing relashionships with ip addresses and name server with phishing

In [19]:
from collections import Counter

all_ips = []

for ips in test_df["ip_addresses"]:
    all_ips.extend(ips)

ip_counts = Counter(all_ips)

shared_ips = {
    ip: count
    for ip, count in ip_counts.items()
    if count > 1
}

print(shared_ips)

{'23.227.38.74': 4, '141.8.197.42': 2, '199.36.158.100': 8, '74.115.51.54': 2, '74.115.51.55': 2}


In [20]:
all_ns = []

for ns_list in test_df["name_servers"]:
    all_ns.extend(ns_list)

ns_counts = Counter(all_ns)

shared_ns = {
    ns: count
    for ns, count in ns_counts.items()
    if count > 1
}

print(shared_ns)

{'ns3.livedns.co.uk': 2, 'ns2.livedns.co.uk': 2, 'ns1.livedns.co.uk': 2, 'ns4.sprinthost.net': 2, 'ns1.sprinthost.ru': 2, 'ns3.sprinthost.net': 2, 'ns2.sprinthost.ru': 2, 'ns-cloud-c1.googledomains.com': 3, 'ns-cloud-c3.googledomains.com': 3, 'ns-cloud-c2.googledomains.com': 3, 'ns-cloud-c4.googledomains.com': 3, 'ns1.googledomains.com': 5, 'ns2.googledomains.com': 5, 'ns3.googledomains.com': 5, 'ns4.googledomains.com': 5, 'ns-510.awsdns-63.com': 2, 'ns-1375.awsdns-43.org': 2, 'ns-522.awsdns-01.net': 2, 'ns-1854.awsdns-39.co.uk': 2}


In [21]:
from collections import defaultdict

ip_urls = defaultdict(list)

for _, row in test_df.iterrows():
    for ip in row["ip_addresses"]:
        ip_urls[ip].append({
            "url": row["URL"],
            "label": row["label"]
        })

In [22]:
for ip, urls in ip_urls.items():
    if len(urls) > 1:
        print("\nIP:", ip)

        for item in urls:
            label_text = (
                "Phishing"
                if item["label"] == 0
                else "Legitimate"
            )

            print(label_text, ":", item["url"])


IP: 23.227.38.74
Legitimate : https://www.dixxon.com
Legitimate : https://www.ibeani.co.uk
Legitimate : https://www.metroretrovintage.com
Legitimate : https://www.olly.com

IP: 141.8.197.42
Phishing : http://www.f0519141.xsph.ru
Phishing : http://www.f0535398.xsph.ru

IP: 199.36.158.100
Phishing : https://service-mitld.firebaseapp.com/
Phishing : https://liuy-9a930.web.app/
Phishing : https://hidok4f8zl.firebaseapp.com/
Phishing : https://mechinchem-5cb8a.web.app/
Phishing : https://fb-restriction-case-97be5.web.app/
Phishing : https://metafb-tvz6efk.web.app/
Phishing : https://ggsexpole.web.app/
Phishing : https://ormvoixregl1.firebaseapp.com/

IP: 74.115.51.54
Phishing : http://att-103731-107123.weeblysite.com/
Phishing : https://att-104164.weeblysite.com/

IP: 74.115.51.55
Phishing : http://att-103731-107123.weeblysite.com/
Phishing : https://att-104164.weeblysite.com/


In [23]:
from collections import defaultdict

ns_urls = defaultdict(list)

for _, row in test_df.iterrows():
    for ns in row["name_servers"]:
        ns_urls[ns].append({
            "url": row["URL"],
            "label": row["label"]
        })

In [24]:
for ns, urls in ns_urls.items():
    if len(urls) > 1:
        print("\nName Server:", ns)

        for item in urls:
            label_text = (
                "Phishing"
                if item["label"] == 0
                else "Legitimate"
            )

            print(label_text, ":", item["url"])


Name Server: ns3.livedns.co.uk
Legitimate : https://www.voicefmradio.co.uk
Legitimate : https://www.tileandstonejournal.com

Name Server: ns2.livedns.co.uk
Legitimate : https://www.voicefmradio.co.uk
Legitimate : https://www.tileandstonejournal.com

Name Server: ns1.livedns.co.uk
Legitimate : https://www.voicefmradio.co.uk
Legitimate : https://www.tileandstonejournal.com

Name Server: ns4.sprinthost.net
Phishing : http://www.f0519141.xsph.ru
Phishing : http://www.f0535398.xsph.ru

Name Server: ns1.sprinthost.ru
Phishing : http://www.f0519141.xsph.ru
Phishing : http://www.f0535398.xsph.ru

Name Server: ns3.sprinthost.net
Phishing : http://www.f0519141.xsph.ru
Phishing : http://www.f0535398.xsph.ru

Name Server: ns2.sprinthost.ru
Phishing : http://www.f0519141.xsph.ru
Phishing : http://www.f0535398.xsph.ru

Name Server: ns-cloud-c1.googledomains.com
Phishing : https://service-mitld.firebaseapp.com/
Phishing : https://hidok4f8zl.firebaseapp.com/
Phishing : https://ormvoixregl1.firebaseap

### Testing with whole dataset

In [26]:
df["domain"] = df["URL"].apply(extract_hostname)
df["registered_domain"] = df["domain"].apply(get_registered_domain)

df["ip_addresses"] = df["domain"].apply(get_ip_addresses)
df["name_servers"] = df["registered_domain"].apply(get_name_servers)

df[
    ["URL", "label", "domain", "ip_addresses", "name_servers"]
].head(20)

,URL,label,domain,ip_addresses,name_servers
0,https://www.southbankmosaics.com,1,www.southbankmosaics.com,[142.93.145.212],"[ns2.namebrightdns.com, ns1.namebrightdns.com]"
1,https://www.uni-mainz.de,1,www.uni-mainz.de,[134.93.178.47],"[b.ns14.net, ns-extern.zdv.Uni-Mainz.DE, d.ns1..."
2,https://www.voicefmradio.co.uk,1,www.voicefmradio.co.uk,"[216.137.52.128, 216.137.52.12, 216.137.52.39,...","[ns2.livedns.co.uk, ns1.livedns.co.uk, ns3.liv..."
3,https://www.sfnmjournal.com,1,www.sfnmjournal.com,"[162.159.140.114, 172.66.0.112]","[ns1.reedelsevier.com, ns2.reedelsevier.com, n..."
4,https://www.rewildingargentina.org,1,www.rewildingargentina.org,"[172.67.178.236, 104.21.31.174]","[lovisa.ns.cloudflare.com, hans.ns.cloudflare...."
5,https://www.globalreporting.org,1,www.globalreporting.org,"[104.20.37.79, 172.66.159.113]","[george.ns.cloudflare.com, coco.ns.cloudflare...."
6,https://www.saffronart.com,1,www.saffronart.com,"[172.67.68.10, 104.26.8.7, 104.26.9.7]","[romina.ns.cloudflare.com, damien.ns.cloudflar..."
7,https://www.nerdscandy.com,1,www.nerdscandy.com,"[104.18.12.41, 104.18.13.41]","[amit.ns.cloudflare.com, fish.ns.cloudflare.com]"
8,https://www.hyderabadonline.in,1,www.hyderabadonline.in,"[104.21.73.182, 172.67.165.62]","[yahir.ns.cloudflare.com, chan.ns.cloudflare.com]"
9,https://www.aap.org,1,www.aap.org,"[104.20.47.105, 172.66.148.195]","[ns1.p201.dns.oraclecloud.net, ns3.p201.dns.or..."


#### URLs with IP

In [27]:
has_ip = df["ip_addresses"].apply(len) > 0

print("URLs with IP:", has_ip.sum())
print("Percentage:", has_ip.mean() * 100)

URLs with IP: 8068
Percentage: 80.67999999999999


#### URLs with NS

In [28]:
has_ns = df["name_servers"].apply(len) > 0

print("URLs with NS:", has_ns.sum())
print("Percentage:", has_ns.mean() * 100)

URLs with NS: 8336
Percentage: 83.36


#### URLs having both

In [29]:
has_both = has_ip & has_ns

print("URLs with both:", has_both.sum())
print("Percentage:", has_both.mean() * 100)

URLs with both: 7937
Percentage: 79.36999999999999


### Save current data list to a seperate file

In [30]:
enriched_df = df.copy()
print(enriched_df.shape)
print(enriched_df.columns.tolist())

(10000, 61)
['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label', 'is_phishing', 'domain', 'registered_domain', 'ip_addresses', 'name_s

In [31]:
enriched_df.to_csv(
    "phiusill_enriched_10000.csv",
    index=False
)

In [32]:
df.head(10)

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label,is_phishing,domain,registered_domain,ip_addresses,name_servers
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,28,119,0,124,1,0,www.southbankmosaics.com,southbankmosaics.com,[142.93.145.212],"[ns2.namebrightdns.com, ns1.namebrightdns.com]"
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,8,39,0,217,1,0,www.uni-mainz.de,uni-mainz.de,[134.93.178.47],"[b.ns14.net, ns-extern.zdv.Uni-Mainz.DE, d.ns1..."
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,7,42,2,5,1,0,www.voicefmradio.co.uk,voicefmradio.co.uk,"[216.137.52.128, 216.137.52.12, 216.137.52.39,...","[ns2.livedns.co.uk, ns1.livedns.co.uk, ns3.liv..."
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,15,22,1,31,1,0,www.sfnmjournal.com,sfnmjournal.com,"[162.159.140.114, 172.66.0.112]","[ns1.reedelsevier.com, ns2.reedelsevier.com, n..."
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,34,72,1,85,1,0,www.rewildingargentina.org,rewildingargentina.org,"[172.67.178.236, 104.21.31.174]","[lovisa.ns.cloudflare.com, hans.ns.cloudflare...."
5,23107.txt,https://www.globalreporting.org,30,www.globalreporting.org,23,0,org,100.0,1.000000,0.079963,...,11,86,0,14,1,0,www.globalreporting.org,globalreporting.org,"[104.20.37.79, 172.66.159.113]","[george.ns.cloudflare.com, coco.ns.cloudflare...."
6,23034.txt,https://www.saffronart.com,25,www.saffronart.com,18,0,com,100.0,1.000000,0.522907,...,14,44,2,17,1,0,www.saffronart.com,saffronart.com,"[172.67.68.10, 104.26.8.7, 104.26.9.7]","[romina.ns.cloudflare.com, damien.ns.cloudflar..."
7,696732.txt,https://www.nerdscandy.com,25,www.nerdscandy.com,18,0,com,100.0,1.000000,0.522907,...,22,36,0,15,1,0,www.nerdscandy.com,nerdscandy.com,"[104.18.12.41, 104.18.13.41]","[amit.ns.cloudflare.com, fish.ns.cloudflare.com]"
8,739255.txt,https://www.hyderabadonline.in,29,www.hyderabadonline.in,22,0,in,100.0,1.000000,0.005084,...,9,40,1,317,1,0,www.hyderabadonline.in,hyderabadonline.in,"[104.21.73.182, 172.67.165.62]","[yahir.ns.cloudflare.com, chan.ns.cloudflare.com]"
9,14486.txt,https://www.aap.org,18,www.aap.org,11,0,org,100.0,1.000000,0.079963,...,12,173,6,65,1,0,www.aap.org,aap.org,"[104.20.47.105, 172.66.148.195]","[ns1.p201.dns.oraclecloud.net, ns3.p201.dns.or..."
